# FashionMNIST CNN

This notebook trains a Convolutional Neural Network to classify FashionMNIST images into 10 classes, using a GPU when one is available.

In [ ]:
import copy
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets
from torchvision.transforms import ToTensor

## 1. Select device

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    print('Using GPU:', torch.cuda.get_device_name(0))
else:
    print('Using CPU')

## 2. Load and split data

The 60,000-image training set is split into 54,000 training images and 6,000 validation images. The test set is reserved for final evaluation.

In [ ]:
training_data = datasets.FashionMNIST(root='data', train=True, download=True, transform=ToTensor())
test_data = datasets.FashionMNIST(root='data', train=False, download=True, transform=ToTensor())

train_data, validation_data = random_split(
    training_data, [54000, 6000], generator=torch.Generator().manual_seed(42)
)

batch_size = 64
train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
validation_dataloader = DataLoader(validation_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

## 3. Training and evaluation helpers

In [ ]:
def train_one_epoch(dataloader, model, loss_fn, optimizer):
    model.train()
    total_loss = 0

    for x_batch, y_batch in dataloader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        logits = model(x_batch)
        loss = loss_fn(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)


def test_accuracy(dataloader, model):
    model.eval()
    correct = total = 0

    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            predicted = model(x_batch).argmax(dim=1)
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

    return correct / total

## 4. Build the CNN

Two convolution blocks extract features from each image, and a classifier converts those features into scores for the 10 classes.

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(64 * 7 * 7, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
).to(device)

print('Model device:', next(model.parameters()).device)

## 5. Train with validation

The model weights are saved only when validation accuracy improves.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 5
best_validation_accuracy = 0
model_path = 'best_fashion_cnn_model.pth'

for epoch in range(epochs):
    loss = train_one_epoch(train_dataloader, model, loss_fn, optimizer)
    validation_accuracy = test_accuracy(validation_dataloader, model)

    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        torch.save(model.state_dict(), model_path)

    print(
        f'Epoch {epoch + 1}/{epochs} | loss = {loss:.4f} | '
        f'validation accuracy = {validation_accuracy:.2%}'
    )

## 6. Final test evaluation

In [ ]:
best_model = copy.deepcopy(model)
best_model.load_state_dict(torch.load(model_path, weights_only=True))
best_model = best_model.to(device)

final_test_accuracy = test_accuracy(test_dataloader, best_model)
print(f'Final test accuracy: {final_test_accuracy:.2%}')

## Result

Final test accuracy: **91.15%**.